# PART 1: Prompt Templates

In [43]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("dummy_genai_document.pdf")

documents = loader.load()

print("Number of pages:", len(documents))
print(documents[0].page_content[:500])

Number of pages: 1
Dummy GenAI Knowledge Document
Generative AI is a branch of artificial intelligence that can create new content such as text, code, images, and
summaries. Large Language Models (LLMs) generate text by predicting likely tokens from the context
provided to them.
Retrieval-Augmented Generation (RAG) improves an LLM application by retrieving relevant information from
external documents before generating an answer. A typical RAG pipeline loads documents, splits them into
chunks, creates embeddings, s


In [44]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

Number of chunks: 3


In [45]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings()

vector_store = FAISS.from_documents(
    chunks,
    embeddings
)

print("Documents stored in FAISS.")

Documents stored in FAISS.


## Task 1: PromptTemplate

In [46]:
from langchain_core.prompts import PromptTemplate

In [47]:
prompt = PromptTemplate(
    template="""
You are a helpful Generative AI assistant.

Answer the user's question clearly and simply.

User Question:
{question}

Answer:
""",
    input_variables=["question"]
)

In [48]:
result = prompt.invoke({
    "question": "What is Generative AI?"
})

print(result)

text="\nYou are a helpful Generative AI assistant.\n\nAnswer the user's question clearly and simply.\n\nUser Question:\nWhat is Generative AI?\n\nAnswer:\n"


In [49]:
result = prompt.invoke({
    "question": "What is RAG?"
})

print(result)

text="\nYou are a helpful Generative AI assistant.\n\nAnswer the user's question clearly and simply.\n\nUser Question:\nWhat is RAG?\n\nAnswer:\n"


In [50]:
result = prompt.invoke({
    "question": "Why are prompt templates useful?"
})

print(result)

text="\nYou are a helpful Generative AI assistant.\n\nAnswer the user's question clearly and simply.\n\nUser Question:\nWhy are prompt templates useful?\n\nAnswer:\n"


## TASK 2: ChatPromptTemplate

In [51]:
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

In [52]:
system_message = SystemMessagePromptTemplate.from_template(
    """
You are a Generative AI tutor.

Explain technical concepts in simple language.
Use examples whenever useful.
"""
)

In [53]:
human_message = HumanMessagePromptTemplate.from_template(
    """
Explain the following topic:

{question}
"""
)

In [54]:
chat_prompt = ChatPromptTemplate.from_messages([
    system_message,
    human_message
])

In [55]:
result = chat_prompt.invoke({
    "question": "What is Retrieval Augmented Generation?"
})

print(result)

messages=[SystemMessage(content='\nYou are a Generative AI tutor.\n\nExplain technical concepts in simple language.\nUse examples whenever useful.\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='\nExplain the following topic:\n\nWhat is Retrieval Augmented Generation?\n', additional_kwargs={}, response_metadata={})]


In [56]:
prompt = PromptTemplate(
    template="Answer this question: {question}",
    input_variables=["question"]
)

In [57]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant."),
    ("human", "{question}")
])

# PART 2: STRUCTURED OUTPUT using PYdantic

## Task 3: Pydantic Output Schema

In [58]:
from pydantic import BaseModel, Field

In [59]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

In [60]:
load_dotenv()

True

In [61]:
class Answer(BaseModel):
    answer: str
    confidence: float
    source : str

In [62]:
llm = ChatOpenAI(model = 'gpt-3.5-turbo')

In [63]:
structured_llm = llm.with_structured_output(Answer)

result = structured_llm.invoke('What is RAG?')

c:\Users\arunk\anaconda3\envs\genai_env\Lib\site-packages\langchain_openai\chat_models\base.py:2657: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


In [64]:
print("Answer:", result.answer)
print("Confidence:", result.confidence)
print("Source:", result.source)

Answer: RAG stands for Red, Amber, Green. It is a color coding system used in project management to indicate the status of a project or task. Red typically signifies a project or task that is behind schedule or at risk, Amber signifies a project or task that is progressing but may require attention, and Green signifies a project or task that is on track and progressing as planned.
Confidence: 0.9
Source: Project Management


## TASK 4: Valdiatoin and Error Handlingdef get_structured_answer(question):

In [65]:
def get_structured_answer(question):

    try:
        result = structured_llm.invoke(question)
        answer = Answer.model_validate(result)

        return answer
    except Exception as e:

        print("Invalid LLM output:", e)
        return Answer(
            answer="Unable to generate a valid structured response.",
            confidence=0.0,
            source="Fallback"
        )

In [66]:
result = get_structured_answer('What is generative ai?')

In [67]:
result

Answer(answer='Generative AI refers to artificial intelligence models that are capable of creating new data, such as images, text, or audio, that is similar to the data it has been trained on. These models can generate new content based on patterns and data they have learned during training, without direct human input. Generative AI is often used in applications such as image generation, text generation, and music composition.', confidence=0.9, source='Self')

# Part 3: Chains in LangChain

## TASK 5 : Simple Chain

In [68]:
from langchain_core.prompts import PromptTemplate

In [69]:
llm = ChatOpenAI(model = 'gpt-3.5-turbo')

In [70]:
simple_prompt = ChatPromptTemplate.from_template(
    template="""
    Explain the following topic in simple language:

    {question}
"""
)

simple_chain = simple_prompt | llm

In [71]:
result = simple_chain.invoke({
    "question": "What is RAG?"
})

print(result.content)

RAG stands for Red, Amber, Green. It is a simple way to show the status or level of something. Red means something is bad or needs attention, amber means something is a bit off or could be improved, and green means everything is good or on track. It is often used in project management or to show the progress of tasks or goals.


## TASK 6: Conditional Chain

In [72]:
direct_prompt = ChatPromptTemplate.from_template(
    """
    Answer the following question directly.

    Question:
    {question}
    """
)

direct_chain = direct_prompt | llm

In [73]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

In [74]:
retrieval_prompt = ChatPromptTemplate.from_template(
    """
    Answer the question using only the provided context.

    Context:
    {context}

    Question:
    {question}
    """
)

In [75]:
retrieval_chain = (
    {
        "context": lambda x: retriever.invoke(x["question"]),
        "question": lambda x: x["question"]
    }
    | retrieval_prompt
    | llm
)

In [76]:
def is_factual(question):

    factual_words = [
        "what",
        "who",
        "when",
        "where",
        "define",
        "explain",
        "how does"
    ]
    question = question.lower()
    return any(word in question for word in factual_words)

In [77]:
from langchain_core.runnables import RunnableBranch

conditional_chain = RunnableBranch(
    (
        lambda x: is_factual(x["question"]),
        retrieval_chain
    ),
    direct_chain
)

In [78]:
result = conditional_chain.invoke({
    "question": "What is Retrieval Augmented Generation?"
})

print(result.content)

Retrieval-Augmented Generation (RAG) improves an LLM application by retrieving relevant information from external documents before generating an answer.


In [79]:
result = conditional_chain.invoke({
    "question": "Write a short motivational message."
})

print(result.content)

You are capable of achieving great things. Keep pushing forward, stay focused, and believe in yourself. Your hard work and determination will lead you to success. Keep pushing through the challenges and never give up on your dreams. You are unstoppable.


## TASK 7: Parallel Chain

In [80]:
answer_prompt = ChatPromptTemplate.from_template(
    """
    Answer this question clearly:

    {question}
    """
)

answer_chain = answer_prompt | llm

In [81]:
summary_prompt = ChatPromptTemplate.from_template(
    """
    Give a short summary of the following question/topic:

    {question}
    """
)

summary_chain = summary_prompt | llm

In [82]:
followup_prompt = ChatPromptTemplate.from_template(
    """
    Generate 3 useful follow-up questions about:

    {question}
    """
)

followup_chain = followup_prompt | llm

In [83]:
from langchain_core.runnables import RunnableParallel

In [84]:


parallel_chain = RunnableParallel(
    answer=answer_chain,
    summary=summary_chain,
    followup_questions=followup_chain
)

In [85]:
from langchain_core.runnables import RunnableParallel

In [86]:
result = parallel_chain.invoke({
    "question": "What is Retrieval Augmented Generation?"
})

In [87]:
print("ANSWER:")
print(result["answer"].content)

print("\nSUMMARY:")
print(result["summary"].content)

print("\nFOLLOW-UP QUESTIONS:")
print(result["followup_questions"].content)

ANSWER:
Retrieval Augmented Generation is a machine learning approach that combines two key components: retrieval, which involves searching for relevant information from a large dataset, and generation, which involves generating new content based on the retrieved information. By integrating these two components, the model is able to produce more contextually relevant and accurate outputs.

SUMMARY:
Retrieval Augmented Generation is a technique in natural language processing that combines retrieval-based models with generation-based models to improve the performance of tasks such as text generation and question answering. This approach involves retrieving relevant information from a large database or knowledge base before generating a response or output, allowing the model to incorporate external knowledge and context.

FOLLOW-UP QUESTIONS:
1. How does Retrieval Augmented Generation improve upon traditional text generation models?
2. What are some real-world applications of Retrieval Au

# PART 4 Runnables and LCEL

## Task 8: Runnables Basics

In [88]:
from langchain_core.runnables import RunnablePassthrough

In [89]:
runnable_chain = (
    {
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

In [90]:
result = runnable_chain.invoke(
    "What is Generative AI?"
)

print(result.content)

Generative AI is a type of artificial intelligence that is capable of creating new content and generating new ideas. This technology uses machine learning algorithms to understand and analyze patterns in data, and then uses this knowledge to produce new content, such as images, videos, text, or music. Generative AI can be used in various fields, such as art, music, literature, and design, to create new and original works.


In [91]:
from langchain_core.runnables import RunnablePassthrough

In [92]:
chain = RunnablePassthrough()

result = chain.invoke("Hello LangChain")

print(result)

Hello LangChain


In [93]:
chain = (
    {
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

result = chain.invoke(
    "What is RAG?"
)

print(result.content)

RAG is an acronym that stands for Red, Amber, Green. It is a color-coding system commonly used in project management and reporting to indicate the status or health of a particular project, task, or metric. Red typically indicates that there are significant issues or risks that need urgent attention, amber signifies potential concerns or delays, and green represents that everything is on track or successful.


## TASK 9: LCEL Based RAG Chain

In [94]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

In [95]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_template(
    """
You are a helpful AI assistant.

Answer the question using only the provided context.

If the answer is not available in the context,
say that the information is not available.

Context:
{context}

Question:
{question}

Answer:
"""
)

In [96]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

In [97]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

In [98]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | output_parser
)

In [99]:
answer = rag_chain.invoke(
    "What is Generative AI?"
)

print(answer)

Generative AI is a branch of artificial intelligence that can create new content such as text, code, images, and summaries.


In [100]:
answer = rag_chain.invoke(
    "What is Generative AI?"
)

print(answer)

Generative AI is a branch of artificial intelligence that can create new content such as text, code, images, and summaries. Large Language Models (LLMs) generate text by predicting likely tokens from the context provided to them.


In [101]:
answer = rag_chain.invoke(
    "Why are prompt templates useful?"
)

print(answer)

Prompt templates are useful for LLM applications because they make the applications easier to maintain. Instead of hard-coding every question, a template can contain placeholders such as {question}, which the application fills dynamically at runtime.


## Task 10: Observations and Insights

1. Why structured output is important
- Structured output ensures that the LLM returns data in a predefined format.

2. Advantages of LCEL over traditional chains
- Simple and readable syntax
- Easy composition of components
- Supports sequential pipelines
- Supports parallel execution
- Supports branching and conditional logic
- Works naturally with LangChain Runnables
- Easier to modify individual components

3. When to use parallel vs conditional chains
- Use a conditional chain when only one path should execute.
- Use a parallel chain when multiple operations can execute independently.